In [1]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.ndimage import gaussian_filter

from benchmarks import get_benchmark_speed
from checkpoint_heatmap import heatmap_of_checkpoint, heatmap_slices_of_checkpoint, heatmaps_of_checkpoint_indep
from experiment_sets import set_shared_indep_table, set_shared_indep_table_ring, set_stairs_crawler

def smooth(data):
    return gaussian_filter(data, sigma=3)

/home/erik/.local/lib/python3.10/site-packages/torchrl/data/replay_buffers/samplers.py:34: UserWarning: Failed to import torchrl C++ binaries. Some modules (eg, prioritized replay buffers) may not work with your installation. This is likely due to a discrepancy between your package version and the PyTorch version. Make sure both are compatible. Usually, torchrl majors follow the pytorch majors within a few days around the release. For instance, TorchRL 0.5 requires PyTorch 2.4.0, and TorchRL 0.6 requires PyTorch 2.5.0.
  warnings.warn(EXTENSION_WARNING)


pygame 2.6.1 (SDL 2.28.4, Python 3.10.12)
Hello from the pygame community. https://www.pygame.org/contribute.html


# Heatmaps

In [2]:
from collections import defaultdict
heatmaps_cache = defaultdict(dict)

In [19]:
for run_id in [1603, 1605, 1633, 1621, 1648, 1689]:
    if run_id not in heatmaps_cache:
        heatmaps_cache[run_id][500] = heatmap_of_checkpoint(run_id, 500)
        heatmaps_cache[run_id][1000] = heatmap_of_checkpoint(run_id, 1000)

In [54]:
plt.rcParams.update({'font.size': 22})
fig, axs = plt.subplots(3,4, sharex=True, sharey=True, figsize=(10,7))
cbar_ax = fig.add_axes([.91, .3, .03, .4])

order = [1633, 1648, 1605, 1689, 1603, 1621]
# for run_id in order:
#     print(run_id, set_shared_indep_table.run_metrics[run_id]["training.mean_speed"]["values"][-1])

for i, ax in enumerate(axs.flat):
    run_id = order[i//2]
    t = [500,1000][i%2]
    hm = heatmaps_cache[run_id][t]
    print(np.max(hm), np.min(hm))
    sns.heatmap(hm, ax=ax, annot=False, linewidths=0, cmap='RdBu', vmin=-9, vmax=9, cbar_ax=None if i else cbar_ax, cbar=i == 0)
    ax.set_xticks(np.linspace(0, hm.shape[0], 3))
    ax.set_yticks(np.linspace(0, hm.shape[0], 3))
    ax.set_xticklabels(["", "", ""])
    ax.set_yticklabels(["", "", ""])
    # ax.set_xticklabels(["$-\pi$", "$0$", "$\pi$"])
    # ax.set_yticklabels(["$-\pi$", "$0$", "$\pi$"])
    ax.xaxis.set_label_position("top")
    # if i > 2:
    #     ax.set(xlabel="Angle $i-1$")
    # if i % 3 == 0:
    #     ax.set(ylabel="Angle $i+1$")
axs[0,0].set(xlabel="DDPG\n500 eps")
axs[0,1].set(xlabel="DDPG\n1000 eps")
axs[0,2].set(xlabel="PPO\n500 eps")
axs[0,3].set(xlabel="PPO\n1000 eps")
plt.suptitle("Torque at node $i$ as a function of neighbouring deflections", y=1.04)
plt.savefig("final_plots/heatmaps_ddpg_vs_ppo.png", bbox_inches='tight')
plt.show()

8.99999 -8.99999
8.99999 -8.99999
8.9882345 -8.977675
8.739616 -8.259525
8.99999 -8.99999
8.99999 -8.999986
8.99999 -8.99999
8.365604 -8.809329
8.99999 -8.99999
8.99999 -8.99999
8.999746 -8.99999
8.995784 -8.999121


In [126]:
fig = plt.figure(figsize=(10,8))
x = np.linspace(-np.pi, np.pi, 400)
obs = np.stack(np.meshgrid(x, x))
hm = np.clip(-50 * ( obs[0] - obs[1]), -9, 9)

ax = sns.heatmap(hm, annot=False,  linewidths=0, cmap='RdBu')
ax.set_xticks(np.linspace(0, hm.shape[0], 3))
ax.set_yticks(np.linspace(0, hm.shape[0], 3))
ax.set_xticklabels(["$-\pi$", "$0$", "$\pi$"])
ax.set_yticklabels(["$-\pi$", "$0$", "$\pi$"])
ax.set(xlabel="$\delta\\theta_{i+1}$", ylabel="$\delta\\theta_{i-1}$", title="Torque at node $i$ as a function of neighbouring deflections\nusing simple non-reciprocity with $\kappa^{\\alpha} = -50$\n")
# plt.savefig("final_plots/heatmap_benchmark.png",  bbox_inches='tight')
plt.show()

In [7]:
plt.rcParams.update({'font.size': 12})
def plot_all_and_mean(ax, data, label, i, benchmark=None):
    # settings = settings[0]
    # benchmark_speed = get_benchmark_speed(**settings)
    if benchmark is not None:
        ax.axhline(benchmark, ls='--', color='black', lw=1, zorder=10, label="NR")

    color = plt.rcParams['axes.prop_cycle'].by_key()['color'][i]
    mean = np.mean(data, axis=0)
    x = np.arange(len(mean))
    for y in data:
        ax.plot(x, smooth(y), color=color, alpha=0.2)
    ax.plot(x, smooth(mean), label=label, color=color, alpha=1)

fig, axs = plt.subplots(1,1, figsize=(10,5))
benchmark_speed = get_benchmark_speed("crawler", 10) * 100
plot_all_and_mean(axs, np.array([set_shared_indep_table.run_metrics[i]["training.mean_speed"]["values"] for i in [1603, 1605, 1633]]) * 100, "DDPG", 0, benchmark_speed)
plot_all_and_mean(axs, np.array([set_shared_indep_table.run_metrics[i]["training.mean_speed"]["values"] for i in [1621, 1648, 1689]]) * 100, "PPO", 1)
plt.legend()
plt.xlabel("training episodes")
plt.ylabel("mean speed")
plt.title("Horizontal speed of a 10-node crawler during training")
plt.savefig("final_plots/crawler_10.png",  bbox_inches='tight')
plt.show()


fig, axs = plt.subplots(1,1, figsize=(10,5))
benchmark_speed = get_benchmark_speed("ring", 10) * 100
plot_all_and_mean(axs, np.array([set_shared_indep_table_ring.run_metrics[i]["training.mean_speed"]["values"] for i in [1729, 1747, 1762]]) * 100, "DDPG", 0, benchmark_speed)
plot_all_and_mean(axs, np.array([set_shared_indep_table_ring.run_metrics[i]["training.mean_speed"]["values"] for i in [1717, 1719, 1773]]) * 100, "PPO", 1)
plt.legend()
plt.xlabel("training episodes")
plt.ylabel("mean speed")
plt.title("Horizontal speed of a 10-node ring during training")
plt.savefig("final_plots/ring_10.png",  bbox_inches='tight')
plt.show()


In [12]:
for run_id in [1729, 1747, 1762, 1717, 1719, 1773]:
    if run_id not in heatmaps_cache:
        heatmaps_cache[run_id][100] = heatmap_of_checkpoint(run_id, 100)
        heatmaps_cache[run_id][1000] = heatmap_of_checkpoint(run_id, 1000)

/home/erik/Documents/UvA master AI/jaar 2/Thesis/2025CSLThesis/torchrl/checkpoint_heatmap.py:15: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  saved_params = torch.load(path

Loaded checkpoint 100 of run 1729:
- scenario: ring
- algorithm: ddpg
- n_particles: 10
- observation_func: dth_neighbours
- terrain_type: flat
- terrain_settings: None
- policy_net_config: {'depth': 2, 'num_cells': 256}
- share_parameters_policy: True
Finished loading
asserted
before sample
after sample
Loaded checkpoint 1000 of run 1729:
- scenario: ring
- algorithm: ddpg
- n_particles: 10
- observation_func: dth_neighbours
- terrain_type: flat
- terrain_settings: None
- policy_net_config: {'depth': 2, 'num_cells': 256}
- share_parameters_policy: True


/home/erik/Documents/UvA master AI/jaar 2/Thesis/2025CSLThesis/torchrl/checkpoint_heatmap.py:15: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  saved_params = torch.load(path

Finished loading
asserted
before sample
after sample
Loaded checkpoint 100 of run 1747:
- scenario: ring
- algorithm: ddpg
- n_particles: 10
- observation_func: dth_neighbours
- terrain_type: flat
- terrain_settings: None
- policy_net_config: {'depth': 2, 'num_cells': 256}
- share_parameters_policy: True


/home/erik/Documents/UvA master AI/jaar 2/Thesis/2025CSLThesis/torchrl/checkpoint_heatmap.py:15: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  saved_params = torch.load(path

...

Finished loading
asserted
before sample
after sample
Loaded checkpoint 100 of run 1773:
- scenario: ring
- algorithm: ppo
- n_particles: 10
- observation_func: dth_neighbours
- terrain_type: flat
- terrain_settings: None
- policy_net_config: {'depth': 2, 'num_cells': 256}
- share_parameters_policy: True


/home/erik/Documents/UvA master AI/jaar 2/Thesis/2025CSLThesis/torchrl/checkpoint_heatmap.py:15: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  saved_params = torch.load(path

Finished loading
asserted
before sample
after sample
Loaded checkpoint 1000 of run 1773:
- scenario: ring
- algorithm: ppo
- n_particles: 10
- observation_func: dth_neighbours
- terrain_type: flat
- terrain_settings: None
- policy_net_config: {'depth': 2, 'num_cells': 256}
- share_parameters_policy: True


/home/erik/Documents/UvA master AI/jaar 2/Thesis/2025CSLThesis/torchrl/checkpoint_heatmap.py:15: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  saved_params = torch.load(path

Finished loading
asserted
before sample
after sample


In [14]:
plt.rcParams.update({'font.size': 22})
fig, axs = plt.subplots(3,4, sharex=True, sharey=True, figsize=(10,7))
cbar_ax = fig.add_axes([.91, .3, .03, .4])

order = [1762, 1719, 1729, 1717, 1747, 1773]
# for run_id in order:
#     print(run_id, set_shared_indep_table.run_metrics[run_id]["training.mean_speed"]["values"][-1])

for i, ax in enumerate(axs.flat):
    run_id = order[i//2]
    t = [100,1000][i%2]
    hm = heatmaps_cache[run_id][t]
    print(np.max(hm), np.min(hm))
    sns.heatmap(hm, ax=ax, annot=False, linewidths=0, cmap='RdBu', vmin=-9, vmax=9, cbar_ax=None if i else cbar_ax, cbar=i == 0)
    ax.set_xticks(np.linspace(0, hm.shape[0], 3))
    ax.set_yticks(np.linspace(0, hm.shape[0], 3))
    ax.set_xticklabels(["", "", ""])
    ax.set_yticklabels(["", "", ""])
    # ax.set_xticklabels(["$-\pi$", "$0$", "$\pi$"])
    # ax.set_yticklabels(["$-\pi$", "$0$", "$\pi$"])
    ax.xaxis.set_label_position("top")
    # if i > 2:
    #     ax.set(xlabel="Angle $i-1$")
    # if i % 3 == 0:
    #     ax.set(ylabel="Angle $i+1$")
axs[0,0].set(xlabel="DDPG\n100 eps")
axs[0,1].set(xlabel="DDPG\n1000 eps")
axs[0,2].set(xlabel="PPO\n100 eps")
axs[0,3].set(xlabel="PPO\n1000 eps")
plt.suptitle("Torque at node $i$ as a function of neighbouring deflections", y=1.04)
plt.savefig("final_plots/heatmaps_ddpg_vs_ppo_ring.png", bbox_inches='tight')
plt.show()

8.968268 -8.9999075
8.99999 -8.99999
8.952574 -8.957967
8.99999 -8.99999
8.985695 -8.999418
8.99999 -8.99999
8.045671 -7.618958
8.99999 -8.99999
8.364663 -8.999983
8.99999 -8.99999
6.622637 -8.299512
8.99999 -8.99999


# Heatmap slices

In [9]:
hms = heatmap_slices_of_checkpoint(2047, 650, -20, 20, 9)

/home/erik/Documents/UvA master AI/jaar 2/Thesis/2025CSLThesis/torchrl/checkpoint_heatmap.py:15: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  saved_params = torch.load(path

Loaded checkpoint 650 of run 2047:
- scenario: ring
- algorithm: ddpg
- n_particles: 10
- observation_func: dth_neighbours_plus_thdot
- terrain_type: mesh
- terrain_settings: flat
- policy_net_config: {'depth': 2, 'num_cells': 256}
- share_parameters_policy: True
Finished loading
torch.Size([400, 400, 10, 3])
torch.Size([400, 400, 10, 3])
torch.Size([400, 400, 10, 3])
torch.Size([400, 400, 10, 3])
torch.Size([400, 400, 10, 3])
torch.Size([400, 400, 10, 3])
torch.Size([400, 400, 10, 3])
torch.Size([400, 400, 10, 3])
torch.Size([400, 400, 10, 3])


In [10]:
plt.rcParams.update({'font.size': 12})
from matplotlib import transforms
from matplotlib.lines import Line2D

x = np.linspace(-np.pi, np.pi, len(hms[0]))
y = np.linspace(-np.pi, np.pi, len(hms[0]))
X, Y = np.meshgrid(x, y)

fig, ax = plt.subplots(figsize=(12, 4))

for i in range(len(hms)):
    Z = hms[i][::-1,:]
    im = ax.imshow(Z, extent=[-3, 3, -3, 3],
                   origin='lower', cmap='RdBu', alpha=1, vmin=-9, vmax=9)

    # Apply a shear + translation transform
    base = ax.transData
    shear = transforms.Affine2D().skew_deg(xShear=0, yShear=22.5).scale(sx=0.5, sy=1).translate(i*3.0, 0)
    im.set_transform(shear + base)

ax.set_xlim(-4, 28)
ax.set_ylim(-5, 7)
ax.axis('off')

t = transforms.Affine2D().skew_deg(xShear=0, yShear=22.5).scale(sx=0.5, sy=1).translate(i*3.0, 0)
x_ticks = [-np.pi, 0, np.pi]
x_tick_labels = ["$-\pi$", "$0$", "$\pi$"]

for i in range(3):
    x = x_ticks[i]
    line = Line2D([x, x], [-np.pi, -np.pi-0.3], color='black', transform=t + ax.transData)
    ax.add_line(line)
    ax.text(x, -3.7, x_tick_labels[i], ha='center', va='top', transform=t + ax.transData)
ax.text(0, -4.5, "$\delta\\theta_{i+1}$", ha='center', va='top', transform=t + ax.transData, rotation=45, rotation_mode="anchor")

for i in range(3):
    x = x_ticks[i]
    line = Line2D([np.pi, np.pi+0.3], [x, x], color='black', transform=t + ax.transData)
    ax.add_line(line)
    ax.text(3.7, x, x_tick_labels[2-i], ha='left', va='center', transform=t + ax.transData)
ax.text(5.5, -1.5, "$\delta\\theta_{i-1}$", ha='left', va='center', transform=t + ax.transData, rotation=90, rotation_mode="anchor")

z_ticks = np.linspace(-20, 20, 9)
for i in range(9):
    z = 3 * i
    line = Line2D([z, z], [4, 4.5], color='black')
    ax.add_line(line)
    ax.text(z, 4.7, "{:.2f}".format(z_ticks[i]), ha='center', va='bottom')
ax.text(12, 5.5, "$\dot\\theta_i$", ha='center', va='bottom')

plt.title("Torque at node $i$ as a function of neighbouring deflections and rotational velocity")
plt.savefig("final_plots/thdot_heatmap_tunnel_flat.png",  bbox_inches='tight')

In [50]:
# from mpl_toolkits.mplot3d import Axes3D

# fig = plt.figure(figsize=(8, 6))
# ax = fig.add_subplot(111, projection='3d')

# z_slices = np.linspace(0, 2*np.pi, 6)
# x = np.linspace(-np.pi, np.pi, len(hms[0]))
# y = np.linspace(-np.pi, np.pi, len(hms[0]))
# X, Y = np.meshgrid(x, y)
# for z in z_slices:
#     Z = hms[0]
#     ax.contourf(X, Y, Z, zdir='x', offset=z, cmap='RdBu', levels=20, alpha=1)

# ax.set_xlabel("x")
# ax.set_ylabel("y")
# ax.set_zlabel("z")

# plt.show()

In [51]:
# from mpl_toolkits.mplot3d import Axes3D  # activates 3D plotting

# # Example 3-input function
# def f(x, y, z):
#     return np.sin(x) * np.cos(y) + np.sin(z)

# # Create grid
# x = np.linspace(-3, 3, 100)
# y = np.linspace(-3, 3, 100)
# X, Y = np.meshgrid(x, y)
# z_slices = np.linspace(0, 2*np.pi, 6)

# fig = plt.figure(figsize=(9, 4))
# ax = fig.add_subplot(111, projection='3d')

# # Disable perspective (orthographic projection)
# ax.set_proj_type('ortho')

# # Draw stacked heatmaps
# for z in z_slices:
#     Z = f(X, Y, z)
#     ax.plot_surface(
#         X, Y, z*np.ones_like(Z),
#         rstride=1, cstride=1,
#         facecolors=plt.cm.RdYlBu((Z - Z.min()) / (Z.max() - Z.min())),
#         shade=False
#     )

# ax.set_xlabel('x')
# ax.set_ylabel('y')
# ax.set_zlabel('z')
# ax.view_init(elev=10, azim=-70)  # tune to match your sketch angle

# plt.show()

# heatmaps terrain

# Indep heatmaps

In [3]:
hms_crawler = heatmaps_of_checkpoint_indep(1686, 1000)
hms_ring = heatmaps_of_checkpoint_indep(1748, 1000)

/home/erik/Documents/UvA master AI/jaar 2/Thesis/2025CSLThesis/torchrl/checkpoint_heatmap.py:15: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  saved_params = torch.load(path

Loaded checkpoint 1000 of run 1686:
- scenario: crawler
- algorithm: ddpg
- n_particles: 10
- observation_func: dth_neighbours
- terrain_type: flat
- terrain_settings: None
- policy_net_config: {'depth': 2, 'num_cells': 256}
- share_parameters_policy: False
Finished loading
asserted
before sample


/home/erik/.local/lib/python3.10/site-packages/torch/functional.py:534: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at ../aten/src/ATen/native/TensorShape.cpp:3595.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


after sample
Loaded checkpoint 1000 of run 1748:
- scenario: ring
- algorithm: ddpg
- n_particles: 10
- observation_func: dth_neighbours
- terrain_type: flat
- terrain_settings: None
- policy_net_config: {'depth': 2, 'num_cells': 256}
- share_parameters_policy: False


/home/erik/Documents/UvA master AI/jaar 2/Thesis/2025CSLThesis/torchrl/checkpoint_heatmap.py:15: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  saved_params = torch.load(path

Finished loading
asserted
before sample
after sample


In [38]:
plt.rcParams.update({'font.size': 20})

fig, axs = plt.subplots(2,10,figsize=(20, 4))
cbar_ax = fig.add_axes([0.91, 0.15, 0.015, 0.7])
sns.heatmap(hms_crawler[7], ax=axs[0,8], annot=False, linewidths=0, cmap='RdBu', vmin=-9, vmax=9, cbar=True, cbar_ax=cbar_ax)
for i in range(10):
    if i in range(1,8):
        sns.heatmap(hms_crawler[i-1], ax=axs[0,i], annot=False, linewidths=0, cmap='RdBu', vmin=-9, vmax=9, cbar=False)
    hm = sns.heatmap(hms_ring[i], ax=axs[1,i], annot=False, linewidths=0, cmap='RdBu', vmin=-9, vmax=9, cbar=False)
    for j in range(2):
        if not (j == 0 and not i in range(1,9)):
            axs[j,i].set_xticks(np.linspace(0, hms_ring[i].shape[0], 3))
            axs[j,i].set_yticks(np.linspace(0, hms_ring[i].shape[0], 3))
            axs[j,i].set_xticklabels(["", "", ""])
            axs[j,i].set_yticklabels(["", "", ""])
        else:
            axs[j,i].axis('off')
    axs[0,i].set_title(f"$i={i}$")
axs[1,0].text(-100,-300,"Crawler",rotation='vertical',va='center',ha='center', size=25)
axs[1,0].text(-100,200,"Ring",rotation='vertical',va='center',ha='center', size=30)
plt.suptitle("Torque at node $i$ as a function of neighbouring deflections", y=1.07)
plt.savefig("final_plots/heatmaps_indep.png",  bbox_inches='tight')
plt.show()

# THDOT

In [6]:
plt.rcParams.update({'font.size': 12})


fig, axs = plt.subplots(2,2, sharex=True, figsize=(15,7))
for row_axes in axs:
    row_axes[1].sharey(row_axes[0])
i = 0
datasets = {
    "crawler": set_shared_indep_table,
    "ring": set_shared_indep_table_ring
}
for shape in ["crawler", "ring"]:
    for algorithm in ["ddpg", "ppo"]:
        data = datasets[shape].get_speed({'n_particles': 10, 'algorithm': algorithm, 'share_parameters_policy': True})
        j = 0
        ax = axs.flat[i]
        ax.axhline(get_benchmark_speed(shape, 10) * 100, ls='--', color='black', lw=1, zorder=10, label="NR")
        for d in data:
            label = "With $\dot\\theta$" if d["label"]["observation_func"] == "dth_neighbours_plus_thdot" else "Without $\dot\\theta$"
            plot_all_and_mean(ax, np.array(d["speeds"])*100, label, j)
        
            ax.grid(True)
            ax.set_title(f"10-node {shape}, {algorithm.upper()}")
            j += 1

        i += 1

for i in range(2):
    axs[1,i].set_xlabel("training episodes")
for i in range(2):
    axs[i,0].set_ylabel("mean speed")
handles, labels = axs[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc='center left', bbox_to_anchor=(0.94, 0.5), borderaxespad=0.)
# plt.tight_layout(rect=[0, 0, 0.85, 1])  # Make room on the right for the legend
plt.suptitle("Horizontal speed of 10-node crawlers and rings during training\nfor observation functions with or without $\dot\\theta$")

plt.savefig("final_plots/thdot.png",  bbox_inches='tight')
plt.show()

# Training

In [16]:
def smooth(data):
    return gaussian_filter(data, sigma=1.5)

plt.rcParams.update({'font.size': 12})
fig = plt.figure()
ax = plt.gca()
ax.axhline(get_benchmark_speed("crawler", 13, "mesh", "stairs") * 100, ls='--', color='black', lw=1, zorder=10, label="NR")
for i, algorithm in enumerate(["ddpg", "ppo"]):
    data = np.array(set_stairs_crawler.get_speed({'algorithm': algorithm})[0]['speeds']) * 100
    plot_all_and_mean(ax, data, algorithm.upper(), i)
ax.set_xlabel("training episodes")
ax.set_ylabel("mean speed")
ax.set_title("Horizontal speed of 13-node crawlers climbing stairs during training")
plt.legend()
plt.savefig("final_plots/crawler_stairs_training.png",  bbox_inches='tight')
plt.show()

In [2]:
plt.rcParams.update({'font.size': 12})



fig, axs = plt.subplots(2,3, sharex=True, sharey=True, figsize=(15,7))
i = 0
for obs_func in ["dth_neighbours", "dth_neighbours_plus_thdot"]:
    for n_particles in [5, 10, 15]:
        data = set_shared_indep_table.get_speed({'n_particles': n_particles, 'observation_func': obs_func, 'share_parameters_policy': False})
        j = 0
        for d in data:
            ax = axs.flat[i]
            plot_all_and_mean(ax, np.array(d["speeds"]), d["label"], j)
            ax.yaxis.set_label_position("right")
            ax.yaxis.tick_right()
        
            ax.grid(True)
            ax.legend()
            j += 1

        i += 1
plt.show()

NameError: name 'plot_all_and_mean' is not defined